# Main program testing (with Yolo pretrained model included)
### Capgemini Engineering
##### Developed by : EL HANNACH Walid

## Overview

This notebook implements our driver drowsiness system on a real video sequence. 
- The model analyzes video footage to detect if the driver is drowsy or not based on our previous "yolov8n" pretrained model, we have impoted it's weight file "best.pt".
- If the program detect that the driver is drowsy we use the modules that we have developed previousely for "perclos calculation" and "posture deviation" to detect the level of drowsiness.
- The YOLOv8 model is tested on Google Colab for faster processing because our laptop cannot handle real-time processing of the videos frames due to GPU inexistance.
- The model has been tested on a video sequence because google colab cannot acces to our webcam or create an external windows.


### Cell 1: Install Required Libraries

- ultralytics: Provides the YOLOv8 model for real-time object detection in video frames.
- opencv-python: Handles image processing tasks, including reading, writing, and modifying video frames.
- dlib: Facilitates facial landmark detection and PERCLOS calculation by providing tools for analyzing facial features.
- mediapipe: Assists in posture analysis by offering robust, real-time tracking of body landmarks such as head position.

In [ ]:
!pip install ultralytics opencv-python dlib mediapipe


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.3 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.

### Cell 2: Load Model, Process Video, and Detect Drowsiness

- Imports the necessary libraries and custom modules.
- Loads the YOLOv8 model and video file.
- Processes each video frame to detect drowsiness level by calculating PERCLOS and head deviation angles.
- Annotates the video with drowsiness levels returned and saves the output video.

In [ ]:
import cv2 # OpenCV for video processing and frame manipulation.
import os  # Standard library for interacting with the operating system (not used in this code).
import sys # Standard library for system-specific parameters and functions (not used in this code).
from google.colab import files # Allows file download from Google Colab.
from ultralytics import YOLO # Provides YOLOv8 model for object detection.
from perclos_calculation import PERCLOSCalculator # Calculates PERCLOS for assessing eye closure.
from posture_deviation import process_frame_for_posture # Analyzes head posture to determine drowsiness.
from google.colab import files


model = YOLO('best.pt')  


cap = cv2.VideoCapture('test.mp4')  
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
output_path = '/content/drowsiness_detection_output4.mp4'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))


perclos_calculator = PERCLOSCalculator(fps=fps)


drowsy_detected = False

def evaluate_drowsiness_level(perc, head_angle):

    """
    Description: Determines the drowsiness level based on the PERCLOS value (perc) and head angle deviation (head_angle).
    Parameters:
                - perc (float): Percentage of eye closure (PERCLOS) used to assess drowsiness.
                - head_angle (float): Angle of head deviation used to evaluate drowsiness.
    Returns: A string indicating the drowsiness level: "Level 1", "Level 2", "Level 3", or "None".
    """

    if 20 <= perc < 30 or 10 <= head_angle < 20:
        return "Level 1"
    elif 30 <= perc < 50 or 20 <= head_angle < 40:
        return "Level 2"
    elif perc >= 50 or head_angle >= 40:
        return "Level 3"
    else:
        return "None"

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)


    results = model(frame)


    ear, perc = perclos_calculator.process_frame_for_perclos(gray_frame)
    posture_frame, head_angle = process_frame_for_posture(frame)


    drowsiness_level = evaluate_drowsiness_level(perc, head_angle)


    if drowsiness_level != "None" and not drowsy_detected:
        drowsy_detected = True


    if drowsy_detected:
        cv2.putText(frame, f'PERCLOS: {perc:.2f}%', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f'Head Deviation Angle: {head_angle:.2f}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f'Drowsiness Level: {drowsiness_level}', (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)


    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = model.names[int(box.cls[0])]
            confidence = box.conf[0]

            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame, f'{label} {confidence:.2f}', (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    out.write(frame)

cap.release()
out.release()

print(f"Output video saved at {output_path}")

# Downloads the processed video file to our local machine.

files.download(output_path)



0: 384x640 1 awake, 9.3ms
Speed: 2.4ms preprocess, 9.3ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 9.2ms
Speed: 3.1ms preprocess, 9.2ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)


/usr/local/lib/python3.10/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



0: 384x640 1 awake, 1 drowsy, 9.8ms
Speed: 3.4ms preprocess, 9.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 10.2ms
Speed: 3.3ms preprocess, 10.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 13.7ms
Speed: 3.3ms preprocess, 13.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 9.4ms
Speed: 3.2ms preprocess, 9.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 9.3ms
Speed: 3.2ms preprocess, 9.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 9.6ms
Speed: 3.2ms preprocess, 9.6ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 9.5ms
Speed: 3.8ms preprocess, 9.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 awake, 12.2ms
Speed: 3.5ms preprocess, 12.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>